In [1]:
import pandas as pd
import numpy as np

In [8]:
file_name = 'weather_hanoi_final.csv'
df = pd.read_csv(file_name)

BƯỚC 1: XỬ LÝ VÀ CHUẨN HÓA CỘT

In [9]:
# 1a. Chuẩn hóa cột 'time' về định dạng datetime
df['time'] = pd.to_datetime(df['time'])

In [10]:
# 1b. LÀM SẠCH TÊN CỘT ĐỂ KHÔNG BỊ LỖI KEYERROR
# Loại bỏ các ký tự đặc biệt [() %] và thay thế khoảng trắng bằng gạch dưới.
# Sau đó loại bỏ gạch dưới kép và gạch dưới ở cuối/đầu.
df.columns = df.columns.str.replace(r'[\(\)%]', '', regex=True).str.replace(' ', '_').str.lower()
df.columns = df.columns.str.replace(r'[\(\)°]', '', regex=True).str.replace(' ', '_').str.lower()
df.columns = df.columns.str.replace('__', '_', regex=True).str.strip('_')
df.rename(columns={'temperature_2m_c': 'temperature_2m_c'}, inplace=True) # Đổi tên cho trực quan

print("Tên cột đã làm sạch:")
print(df.columns.tolist())

Tên cột đã làm sạch:
['time', 'temperature_2m_c', 'relative_humidity_2m', 'precipitation_mm', 'wind_speed_10m_km/h', 'surface_pressure_hpa']


BƯỚC 2: TẠO ĐẶC TRƯNG CHU KỲ (SEASONAL & DIURNAL)

In [11]:
# 2a. Tạo đặc trưng Giờ (Chu kỳ Ngày)
df['hour'] = df['time'].dt.hour
hours_in_day = 24
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / hours_in_day)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / hours_in_day)

In [12]:
# 2b. Tạo đặc trưng Tháng (Chu kỳ Năm/Mùa)
df['month'] = df['time'].dt.month
months_in_year = 12
df['month_sin'] = np.sin(2 * np.pi * df['month'] / months_in_year)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / months_in_year)

BƯỚC 3: TẠO BIẾN MỤC TIÊU DỰ ĐOÁN

In [13]:
# 3a. Tạo cột nhị phân: 1 nếu lượng mưa > 0, 0 nếu không.
df['rain_occurred'] = (df['precipitation_mm'] > 0).astype(int)

In [14]:
# 3b. Dịch chuyển (Shift) cột mục tiêu lên 1 hàng (15 phút)
# Mục tiêu (Y) tại thời điểm t sẽ là sự kiện mưa tại t+15 phút
df['will_rain_next_15min'] = df['rain_occurred'].shift(-1)

BƯỚC 4: DỌN DẸP CUỐI CÙNG VÀ LƯU FILE

In [15]:
# 4a. Loại bỏ các cột không cần thiết cho mô hình
# (time, hour, month, rain_occurred trung gian)
columns_to_drop = [ 'precipitation_mm','hour', 'month', 'rain_occurred']
df_cleaned = df.drop(columns=columns_to_drop)

In [16]:
# 4b. Loại bỏ hàng cuối cùng (sau khi shift, hàng cuối sẽ có giá trị NaN cho cột mục tiêu)
df_cleaned.dropna(subset=['will_rain_next_15min'], inplace=True)
df_cleaned['will_rain_next_15min'] = df_cleaned['will_rain_next_15min'].astype(int)

In [17]:
# 4c. Lưu DataFrame đã làm sạch vào file mới
output_file = 'weather_hanoi_cleaned.csv'
df_cleaned.to_csv(output_file, index=False)

In [5]:
import pandas as pd
df = pd.read_csv("weather_hanoi_cleaned.csv")
print(df.head())

   temperature_2m_c  relative_humidity_2m  wind_speed_10m_km/h  \
0              25.9                    95                  2.4   
1              26.0                    95                  2.2   
2              26.2                    95                  2.5   
3              26.4                    95                  2.7   
4              26.5                    95                  3.1   

   precipitation_mm  surface_pressure_hpa  hour_sin  hour_cos  month_sin  \
0               0.2                1007.3  0.000000  1.000000  -0.866025   
1               0.0                1007.0  0.000000  1.000000  -0.866025   
2               0.0                1006.8  0.000000  1.000000  -0.866025   
3               0.0                1006.5  0.000000  1.000000  -0.866025   
4               0.0                1006.2  0.258819  0.965926  -0.866025   

   month_cos  will_rain_next_15min  
0        0.5                     0  
1        0.5                     0  
2        0.5                     0 

In [9]:
print(df.shape[0])
df.isna().sum()

37247


temperature_2m_c        0
relative_humidity_2m    0
wind_speed_10m_km/h     0
precipitation_mm        0
surface_pressure_hpa    0
hour_sin                0
hour_cos                0
month_sin               0
month_cos               0
will_rain_next_15min    0
dtype: int64